# ROCLING 2026 DSA — 情感知識圖譜生成增強：種子策略 × 風格錨定消融

論文重心實驗。六個合成資料條件 × 3 seed = **18 個 run**，A100 約 4.5 小時。
所有合成資料（各 400 篇）已 commit 進 repo，cell 2 `git clone` 直接拿得到。

| 條件 | seed_mode | 風格錨定 | 長度指令 | 檔案 | 回答什麼 |
|---|---|---|---|---|---|
| **N** | 不給種子詞 | ✗ | fixed | `train_aug_N.csv` | 種子詞有沒有用 |
| **A** | 隨機抽詞 | ✗ | fixed | `train_aug_A.csv` | VA 過濾 vs 隨機 |
| **C** | VA 查表（=實驗5） | ✗ | fixed | `train_aug_C.csv` | 基準線 |
| **E** | 圖擴散 G1+G3 | ✗ | fixed | `train_aug_E.csv` | **圖結構有沒有用** |
| **F** | 圖擴散 | ✓ Val | fixed | `train_aug_F.csv` | 風格錨定有沒有用 |
| **F2** | 圖擴散 | ✓ Val | match_real | `train_aug_F2.csv` | 放寬長度後錨定的效果 |

**核心對照**：C vs E（圖結構）、E vs F（錨定）、F vs F2（長度分布）。
**無增強基準線**免訓練：`macbert_s42/s1/s2` 已存在，dev A_PCC≈0.61。

⚠️ 評估紀律（見 `docs/experiments.md` §0：E4/E19 的 val→test 排序翻轉教訓）：
- 每條件 **3 seed**，比的是條件間 mean 差距 vs 條件內 seed std。
- **不要看單一 run 的分數就下結論**；dev 對增強實驗會失真，最終看官方 test。
- 嚴格 batch 32 / lr 2e-5 / 4 epochs，與實驗 4 對齊。


In [ ]:
# 1) 安裝套件 + 確認 GPU
!pip -q install "transformers>=4.40" jieba scikit-learn scipy

import torch, subprocess
print('=' * 60)
if not torch.cuda.is_available():
    print('⚠️  沒有 GPU！請右上角 Select Kernel → Colab → 選 premium GPU runtime')
else:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU：{name}（{vram:.1f} GB）')
print('=' * 60)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)


In [ ]:
# 2) git clone / pull 程式到 Colab runtime（token 用 getpass 輸入，不要寫進 notebook！）
import os, subprocess
from getpass import getpass

BRANCH = 'feat/ensemble-teacher-student-experiments'   # ← 程式所在 branch
REPO_PATH = '/content/repo' if os.path.exists('/content') else 'repo'

if not os.path.exists(REPO_PATH):
    token = getpass('GitHub token（public repo 直接按 Enter）: ').strip()
    prefix = f'{token}@' if token else ''
    !git clone -q -b {BRANCH} https://{prefix}github.com/chen0427ok/DSA-NIFT.git {REPO_PATH}
os.chdir(REPO_PATH)

# repo 已存在（舊 clone）時：切到正確 branch 並拉最新 commit
!git checkout -q {BRANCH}
r = subprocess.run(['git', 'pull', 'origin', BRANCH], capture_output=True, text=True)
print((r.stdout + r.stderr).strip())
if r.returncode != 0:
    token = getpass('git pull 失敗，輸入新的 GitHub token: ').strip()
    !git remote set-url origin https://{token}@github.com/chen0427ok/DSA-NIFT.git
    !git pull origin {BRANCH}

os.makedirs('outputs/preds', exist_ok=True)
print('工作目錄:', os.getcwd(), '| HEAD:',
      subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())
assert os.path.exists('train_v2.py'), '沒拉到程式，檢查 branch / token'
# 六批合成資料都應該在 repo 裡（已 commit，非 gitignore）
import glob
augs = sorted(glob.glob('data/train_aug_[NACEF]*.csv'))
print('找到的增強檔:', [os.path.basename(a) for a in augs])
need = ['data/train_aug_%s.csv' % c for c in ['N','A','C','E','F','F2']]
missing = [p for p in need if not os.path.exists(p)]
assert not missing, '缺增強檔: %s → git pull 沒成功？' % missing
print('✅ 六批合成資料就緒')


## 3) 跑 18 個 run（六條件 × 3 seed）

`train_v2.py --extra_train data/train_aug_X.csv` 會把該批 400 篇併入訓練。
其餘參數全部預設 = 復現實驗 4（L1 詞典融合，batch 32 / lr 2e-5 / 4 epochs）。
每個 run 產出 `outputs/{run}_best.pt`、`outputs/preds/{run}_{dev,val}.csv`、`outputs/{run}_submission.csv`。

**已存在的 run 會自動跳過**（斷線續跑安全）。A100 每個 run 約 15 分鐘。


In [ ]:
# 3) 主迴圈：六條件 × 3 seed
import os, subprocess, time

CONDITIONS = ['N', 'A', 'C', 'E', 'F', 'F2']
SEEDS = [42, 1, 2]

done, skipped = [], []
t0 = time.time()
for cond in CONDITIONS:
    for s in SEEDS:
        run = f'aug_{cond}_s{s}'
        if os.path.exists(f'outputs/{run}_submission.csv'):
            skipped.append(run); print(f'⏭  跳過 {run}（已存在）'); continue
        aug = f'data/train_aug_{cond}.csv'
        print(f'\n{"="*60}\n▶ {run}  ({aug})\n{"="*60}', flush=True)
        rc = subprocess.call([
            'python', 'train_v2.py',
            '--extra_train', aug,
            '--seed', str(s),
            '--run_name', run,
            '--epochs', '4', '--batch_size', '32', '--lr', '2e-5',
        ])
        if rc == 0:
            done.append(run)
        else:
            print(f'❌ {run} 失敗（rc={rc}），繼續下一個')

print(f'\n完成 {len(done)}／跳過 {len(skipped)}，耗時 {(time.time()-t0)/60:.0f} 分')
print('done:', done)


## 4) 彙整 dev 分數（快速看趨勢；⚠️ dev 對增強實驗會失真，最終仍看官方 test）

每條件的 3 seed 算 mean ± std。**要看的是條件間 mean 差距是否大於條件內 std**，
不是單一 run 的分數。


In [ ]:
# 4) dev 四指標 mean±std（六條件）
import glob, os, numpy as np, pandas as pd

def metrics(df):
    out = {}
    for j in ['valence', 'arousal']:
        p, g = df[f'{j}_pred'].values, df[f'{j}_true'].values
        out[f'{j[0].upper()}_MAE'] = np.mean(np.abs(p - g))
        out[f'{j[0].upper()}_PCC'] = np.corrcoef(p, g)[0, 1] if p.std() > 1e-8 else 0.0
    return out

rows = []
for cond in ['N', 'A', 'C', 'E', 'F', 'F2']:
    per = [metrics(pd.read_csv(f)) for f in sorted(glob.glob(f'outputs/preds/aug_{cond}_s*_dev.csv'))]
    if not per:
        continue
    dfm = pd.DataFrame(per)
    rec = {'cond': cond, 'n_seed': len(per)}
    for k in ['V_MAE', 'V_PCC', 'A_MAE', 'A_PCC']:
        rec[k] = f'{dfm[k].mean():.3f}±{dfm[k].std():.3f}'
    rows.append(rec)

print(pd.DataFrame(rows).to_string(index=False))
print('\n對照：無增強基準（macbert_s42/s1/s2）也可用同法算，作為所有條件的 baseline。')
print('⚠️ dev 分數僅供趨勢；C vs E vs F2 的結論必須以官方 test 提交為準。')


## 5) 打包結果拉回本地（斷線即失！跑完務必執行）

拉回後在本地用 `predict.py` 對 test set 推論、`fetch_results.py --pack` 打包提交。
每天 10 次提交額度：18 個 run + 3 個無增強 baseline = 21 次，兩天用完。


In [ ]:
# 5) 打包 outputs（權重 + 預測 + submission）
import shutil, os
zip_path = shutil.make_archive('kg_ablation_results', 'zip', 'outputs')
print('已打包 ->', os.path.abspath(zip_path), f'({os.path.getsize(zip_path)/1024**3:.2f} GB)')
try:
    from google.colab import files
    files.download('kg_ablation_results.zip')
except Exception:
    print('VS Code 模式無彈窗下載：左側檔案總管找 repo/kg_ablation_results.zip 右鍵 Download')
    print('或掛 Google Drive：')
    print('  from google.colab import drive; drive.mount("/content/drive")')
    print('  shutil.copy("kg_ablation_results.zip", "/content/drive/MyDrive/")')


## 6)（可選）只推論、不重訓：把權重帶回本地後對 test set 跑

若只想拉 submission 而非全部權重（省流量），本地執行：
```bash
for cond in N A C E F F2; do
  for s in 42 1 2; do
    python predict.py --ckpt outputs/aug_${cond}_s${s}_best.pt --lex_mode l1 \
        --input ../DSANIDF_TestSet.csv --run_name aug_${cond}_s${s} --split test
  done
done
```
注意 `--lex_mode l1`（這批都是 10 維 L1，與實驗 4 同架構）。
